# E5: HCC-TACE-Seg external test, repaired

**Revision plan (approved 17 Oct 2026), experiment E5.** Answers Reviewer 2 points 2–4 and Reviewer 4 points 2 and 4
("pipeline choices create artefacts the authors then report as findings").

**Step 0 — reproduction gate.** The submitted external pipeline (Week 10g) is re-run with its original settings and must
reproduce the saved external CNN predictions (four-phase CNN 0/103 HCC, single-phase CNN 5/103). If not, the notebook stops.

**Repairs (each fixed in advance, nothing tuned on external data):**
1. **Intensity window.** The submitted run windowed external CT to −25…125 HU (W150/C50). The internal PLC-CECT data
   were released clipped to −200…200 HU and rescaled to 0–255, so the repaired run uses −200…200 HU. Everything else
   in the Week 10g preprocessing is unchanged (1 mm resampling, SEG liver mask, tumour-centred 96³ crop, z-score).
2. **Mixed-scan inputs.** 14 patients have at least one input series in which every slice position appears twice or
   more (two scans mixed in one series; detected from the DICOM catalogue). Primary analysis: the **89 clean patients**.
   All 103 are also reported.
3. **Real clinical data.** Age and sex are taken from the official TCIA clinical file (downloaded below), instead of
   stand-in values. The earlier fold-median and sex = 0 scenarios are reported for comparison.

**Models:** phase-dropout CNN (main CNN, decided before this test) and the original CNN; single-phase CNN; E2 radiomics
without shape features; CNN+radiomics and full fusion refitted on internal out-of-fold predictions.

**Reported:** HCC sensitivity with exact (Clopper–Pearson) 95% CI, predicted-class distribution, mean P(HCC). The cohort is
HCC-only, so no specificity or AUC is computed here (MCT-LTDiag, E6, provides those). Argmax decisions only.

**Version 2 (17 Oct 2026).** Version 1 stopped at Cell 2 because the TCIA file codes sex as numbers. Version 2 reads
the coding from the file's own glossary sheet (`Sex: 1=Male, 2=Female`) and maps it to the internal coding (male = 1,
female = 0). Nothing else changed.

**Runtime:** GPU runtime recommended (L4 is enough). About 30–45 minutes, mostly reading CT files and radiomics.
**Safety:** inputs are only read; all outputs go to a new folder `liver_project/revision_2026-10/E5_<time>/`.

In [ ]:
# Cell 1 — Setup: installs, paths, input checks, output folder
import os, sys, json, hashlib, platform, datetime, re, time, warnings, subprocess, shutil, tarfile
from pathlib import Path
warnings.filterwarnings('ignore')

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount(os.environ['COLAB_DRIVE_MOUNT'])  # see docs/revision_notebooks.md
    ROOT = Path(os.environ['LIVER_PROJECT_ROOT'])
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'SimpleITK', 'monai', 'lightgbm', 'openpyxl'], check=True)
    radiomics_ok = False
    for spec in (['pyradiomics'], ['git+https://github.com/AIM-Harvard/pyradiomics.git']):
        if subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + spec).returncode == 0:
            radiomics_ok = True
            break
    if not radiomics_ok:
        raise RuntimeError('PyRadiomics could not be installed. Send this output for review.')
else:
    ROOT = Path(os.environ['E5_TEST_ROOT'])

import numpy as np
import pandas as pd
import torch
import SimpleITK as sitk
import joblib
from radiomics import featureextractor
from monai.networks.nets import resnet18
from scipy.ndimage import binary_dilation
from scipy.stats import beta

PROJECT_DIR = ROOT / 'liver_project'
DATA_DIR = ROOT / 'liver_data'
REVISION_DIR = PROJECT_DIR / 'revision_2026-10'
E2_DIR = REVISION_DIR / 'E1_E2_20261017T032941Z'
E3_DIR = REVISION_DIR / 'E3_20261017T041700Z'
EXT_DIR = PROJECT_DIR / 'external_validation' / 'HCC-TACE-Seg'
OLD_PREFIX = os.environ['LIVER_HISTORICAL_PREFIX']   # paths stored in the Week-10 manifest (folder moved 7 Sep 2026)
SEED = 42
CLASSES = ['HCC', 'ICC', 'CHCC']
PHASE_ORDER = ['P', 'C1', 'C2', 'C3']

INPUTS = {
    'manifest_v4': (EXT_DIR / 'nifti_v4_liver_adapter' / 'manifests' / 'external_manifest_for_week10_tumor_only_FILTERED_liver_adapter.csv',
                    'e428d3ad7b5928622e51b78c1c3346b9b168ef9e6b6205b00e33a61f9f84e5ad'),
    'dicom_catalog': (EXT_DIR / 'nifti' / 'logs' / 'dicom_catalog.csv', '515453cc76f824b0803a4d885e0065e395198e4ba3521c2df79b21762e395e62'),
    'historical_external': (PROJECT_DIR / 'results' / 'external_HCC_TACE_SEG_liver_adapter_fixed_103' / 'external_predictions_with_phase_audit.csv',
                            'a5e412a63778013130c90bb50ae37591e16e4d85964053d15abf83b1443424fb'),
    'patient_data': (DATA_DIR / 'patient_data.csv', '8cd4eb9405a3987c35d7ba4e92893c474f20b6a2791ab23bac2c47ef5532b185'),
    'split': (PROJECT_DIR / 'splits' / 'train_val_test.json', 'cfbe3cb6f9b1901c3d672042e7918a95051d841d328d8353492f91376617513e'),
    'radiomics_features': (PROJECT_DIR / 'radiomics' / 'radiomics_features.parquet', 'caac3d2cf4f273bd6eebd48154a1dc4bf20adbbb7a896b3c4947a910fd01e156'),
    'e2_oof_meta': (E2_DIR / 'E2_fusion_oof_meta_features_noshape.csv', 'c20d23d3d0a7df871212f6f937bc2024e686c11dc6975350329cbe186f058d12'),
    'e2_noshape_test': (E2_DIR / 'E2_radiomics_noshape_test_predictions.csv', '9aa460d2ebe597b8b116006af9895fbf9f6fe55f87a9268bef69d69f2830db90'),
    'e3_d_predictions': (E3_DIR / 'E3B_D_phase_dropout_predictions.csv', '4c1aed86c7e8095390b8c4313cb69e12c31bd87a6e67a518e23e3e3e95470ee6'),
}
for f in range(5):
    INPUTS[f'fold_{f}'] = (PROJECT_DIR / 'splits' / f'fold_{f}.json', None)
MODEL_FILES = {
    **{f'w3_{f}': PROJECT_DIR / 'checkpoints_w3' / f'fold_{f}_best.pt' for f in range(5)},
    **{f'w4_{f}': PROJECT_DIR / 'checkpoints_w4' / f'fold_{f}_best.pt' for f in range(5)},
    **{f'd_{f}': E3_DIR / 'D_phase_dropout' / f'fold_{f}.pt' for f in range(5)},
    **{f'rad_{f}': E2_DIR / 'e2_radiomics_noshape_fold_models' / f'fold_{f}.joblib' for f in range(5)},
}

def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1 << 20), b''):
            digest.update(block)
    return digest.hexdigest()

problems, input_hashes = [], {}
for key, (path, expected) in INPUTS.items():
    if not path.exists():
        problems.append(f'missing: {path}')
        continue
    input_hashes[key] = sha256_of(path)
    if expected and input_hashes[key] != expected:
        problems.append(f'changed: {path.name}')
problems += [f'missing: {p}' for p in MODEL_FILES.values() if not p.exists()]
if problems:
    raise RuntimeError('STOP — input check failed:\n' + '\n'.join(problems))

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
import sklearn, scipy, lightgbm, radiomics, monai
VERSIONS = {'python': platform.python_version(), 'torch': torch.__version__, 'monai': monai.__version__,
            'SimpleITK': sitk.Version_VersionString() if hasattr(sitk, 'Version_VersionString') else 'unknown',
            'pyradiomics': getattr(radiomics, '__version__', 'unknown'), 'lightgbm': lightgbm.__version__,
            'scikit-learn': sklearn.__version__, 'numpy': np.__version__, 'device': str(DEVICE)}
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT_DIR = REVISION_DIR / f'E5_{RUN_ID}'
OUT_DIR.mkdir(parents=False, exist_ok=False)
(OUT_DIR / 'inputs').mkdir()
print('Inputs and models checked:', len(INPUTS) + len(MODEL_FILES))
print('Versions:', VERSIONS)
print('Output folder:', OUT_DIR.name)

In [ ]:
# Cell 2 — Download the official TCIA clinical file and extract age and sex (stops if the format is unclear)
CLINICAL_URL = 'https://www.cancerimagingarchive.net/wp-content/uploads/HCC-TACE-Seg_clinical_data-V2.xlsx'
if IN_COLAB:
    import urllib.request
    clinical_path = OUT_DIR / 'inputs' / 'HCC-TACE-Seg_clinical_data-V2.xlsx'
    request = urllib.request.Request(CLINICAL_URL, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(request, timeout=120) as response:
        clinical_path.write_bytes(response.read())
    sheets = pd.read_excel(clinical_path, sheet_name=None)
else:
    clinical_path = OUT_DIR / 'inputs' / 'clinical_test.csv'
    shutil.copy2(os.environ['E5_TEST_CLINICAL'], clinical_path)
    sheets = {'test': pd.read_csv(clinical_path)}
    if os.environ.get('E5_TEST_GLOSSARY'):
        sheets['glossary'] = pd.read_csv(os.environ['E5_TEST_GLOSSARY'])
input_hashes['tcia_clinical'] = sha256_of(clinical_path)

def normalized(name):
    return re.sub(r'[^a-z0-9]+', ' ', str(name).lower()).strip()

def find_clinical_table(sheets):
    for sheet_name, table in sheets.items():
        for column in table.columns:
            values = table[column].dropna().astype(str).str.strip()
            if len(values) and values.str.fullmatch(r'HCC_\d+').mean() > 0.9:
                return sheet_name, table, column
    return None, None, None

sheet_name, table, id_col = find_clinical_table(sheets)
age_cols = [c for c in (table.columns if table is not None else []) if re.search(r'\bage\b', normalized(c))]
sex_cols = [c for c in (table.columns if table is not None else []) if re.search(r'\b(sex|gender)\b', normalized(c))]
print('Sheets:', list(sheets))
if table is not None:
    print('Using sheet:', sheet_name, '| ID column:', id_col, '| age columns:', age_cols, '| sex columns:', sex_cols)
if table is None or len(age_cols) != 1 or len(sex_cols) != 1:
    for name, t in sheets.items():
        print(f'--- sheet {name}: columns', list(t.columns))
    raise RuntimeError('STOP — could not identify a unique patient-ID, age and sex column. '
                       'Send this output; the column choice will be set explicitly.')

SEX_MAP = {'m': 1.0, 'male': 1.0, 'f': 0.0, 'female': 0.0}     # internal coding: male = 1, female = 0

def sex_tokens(series):
    return series.astype(str).str.strip().str.lower().str.replace(r'^(\d+)\.0$', r'\1', regex=True)

def sex_codes_from_glossary(sheets, sex_col):
    """Numeric codes defined in the file's own glossary sheet (TCIA V2: 'Sex | 1=Male, 2=Female')."""
    for name, t in sheets.items():
        if name == sheet_name:
            continue
        for row in [list(t.columns)] + t.astype(str).values.tolist():
            cells = [str(v).strip() for v in row]
            if cells and normalized(cells[0]) == normalized(sex_col):
                text = ' '.join(cells[1:])
                pairs = re.findall(r'(\d+)\s*=\s*(male|female)\b', text, flags=re.I)
                if pairs:
                    return {code: SEX_MAP[word.lower()] for code, word in pairs}, f'sheet {name!r}: {cells[0]} | {text}'
    return {}, None

raw_sex = sex_tokens(table[sex_cols[0]].dropna())
print('Sex values in the file:', raw_sex.value_counts().to_dict())
SEX_CODING = 'text (male/female)'
if set(raw_sex) - set(SEX_MAP):
    glossary_codes, glossary_line = sex_codes_from_glossary(sheets, sex_cols[0])
    print('Sex coding stated in the file glossary:', glossary_line)
    if sorted(glossary_codes.values()) != [0.0, 1.0]:
        raise RuntimeError('STOP — numeric sex codes and no clear male/female definition in the glossary. Send this output.')
    SEX_MAP = {**SEX_MAP, **glossary_codes}
    SEX_CODING = f'numeric, from the glossary ({glossary_line}); mapped to male = 1, female = 0'
unmapped = sorted(set(raw_sex) - set(SEX_MAP))
if unmapped:
    raise RuntimeError(f'STOP — unrecognised sex codes {unmapped}. Send this output; the coding will be set explicitly.')
mapped = raw_sex.map(SEX_MAP)
print(f'Sex after mapping to the internal coding: male {int((mapped == 1).sum())}, female {int((mapped == 0).sum())}')
raw_age = pd.to_numeric(table[age_cols[0]], errors='coerce')
print(f'Age: {raw_age.notna().sum()} numeric values, range {raw_age.min()}–{raw_age.max()}')
if raw_age.notna().sum() and not (0 < raw_age.min() and raw_age.max() < 110):
    raise RuntimeError('STOP — age values outside 0–110. Send this output.')
clinical = pd.DataFrame({
    'patient_id': table[id_col].astype(str).str.strip().str.replace(r'HCC_(\d+)', lambda m: f'HCC_{int(m.group(1)):03d}', regex=True),
    'age': pd.to_numeric(table[age_cols[0]], errors='coerce'),
    'sex': sex_tokens(table[sex_cols[0]]).map(SEX_MAP),
}).dropna(subset=['patient_id'])
conflicts = clinical.groupby('patient_id').agg(age_n=('age', 'nunique'), sex_n=('sex', 'nunique'))
conflicting_ids = set(conflicts.index[(conflicts['age_n'] > 1) | (conflicts['sex_n'] > 1)])
clinical = clinical.groupby('patient_id', as_index=False).first()
clinical.loc[clinical['patient_id'].isin(conflicting_ids), ['age', 'sex']] = np.nan
print(f'Clinical rows: {len(clinical)} patients; conflicting duplicate rows set to missing: {len(conflicting_ids)}')

In [ ]:
# Cell 3 — External manifest (paths remapped to the new folder) and mixed-scan detection
manifest = pd.read_csv(INPUTS['manifest_v4'][0])
path_cols = [c for c in manifest.columns if re.match(r'(ct|mask|liver_mask)_(P|C1|C2|C3)$', c)]
def remap(value):
    if isinstance(value, str) and value.startswith(OLD_PREFIX):
        return str(PROJECT_DIR) + value[len(OLD_PREFIX):]
    return value
for column in path_cols:
    manifest[column] = manifest[column].map(remap)
referenced = {v for c in path_cols for v in manifest[c].dropna() if isinstance(v, str) and v.strip()}
missing_files = [v for v in referenced if not Path(v).exists()]
assert len(manifest) == 103, len(manifest)
if missing_files:
    raise RuntimeError(f'STOP — {len(missing_files)} manifest files not found, e.g. {missing_files[:3]}')

catalog = pd.read_csv(INPUTS['dicom_catalog'][0], usecols=['SeriesInstanceUID', 'Modality', 'ImagePositionPatient'])
catalog = catalog[catalog['Modality'] == 'CT'].copy()
catalog['z'] = catalog['ImagePositionPatient'].map(
    lambda s: round(float(re.findall(r'[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?', s)[2]), 1) if isinstance(s, str) else np.nan)
per_series = catalog.groupby('SeriesInstanceUID').agg(n=('z', 'size'), unique_z=('z', 'nunique'))
per_series['repeats'] = per_series['n'] / per_series['unique_z']

def series_uid(path):
    match = re.search(r'__([0-9.]+)__CT', str(path))
    return match.group(1) if match else None

mixed_rows = []
for _, row in manifest.iterrows():
    for phase in PHASE_ORDER:
        uid = series_uid(row.get(f'ct_{phase}'))
        if uid and uid in per_series.index and per_series.loc[uid, 'repeats'] >= 1.9:
            mixed_rows.append({'patient_id': row['patient_id'], 'channel': phase, 'series_uid': uid,
                               'files': int(per_series.loc[uid, 'n']), 'unique_positions': int(per_series.loc[uid, 'unique_z']),
                               'repeats': float(per_series.loc[uid, 'repeats'])})
mixed = pd.DataFrame(mixed_rows)
MIXED_IDS = set(mixed['patient_id']) if len(mixed) else set()
ALL_IDS = manifest['patient_id'].astype(str).tolist()
CLEAN_IDS = [p for p in ALL_IDS if p not in MIXED_IDS]
mixed.to_csv(OUT_DIR / 'E5_mixed_scan_inputs.csv', index=False)
print(f'Mixed-scan inputs: {len(mixed)} in {len(MIXED_IDS)} patients -> clean cohort {len(CLEAN_IDS)} patients')
if IN_COLAB and (len(mixed), len(MIXED_IDS)) != (16, 14):
    raise RuntimeError('STOP — mixed-scan count differs from the 17 Oct audit (16 inputs, 14 patients). Send this output.')

clinical_ext = pd.DataFrame({'patient_id': ALL_IDS}).merge(clinical, on='patient_id', how='left')
print(f'Observed age for {clinical_ext["age"].notna().sum()}/103, observed sex for {clinical_ext["sex"].notna().sum()}/103 '
      f'(male {int((clinical_ext["sex"] == 1).sum())}, female {int((clinical_ext["sex"] == 0).sum())})')
clinical_ext.to_csv(OUT_DIR / 'E5_external_clinical_joined.csv', index=False)

In [ ]:
# Cell 4 — Preprocessing: Week 10g functions unchanged, run with the historical and the corrected window
TARGET_SPACING_MM = (1.0, 1.0, 1.0)
CROP_SIZE = (96, 96, 96)
TUMOR_DILATION_MM = 15

def resample_to_isotropic(arr, original_spacing, target_spacing=TARGET_SPACING_MM, is_mask=False):   # Week 10g
    img = sitk.GetImageFromArray(arr)
    img.SetSpacing(tuple(original_spacing))
    new_size = [int(round(sz * os_ / ts_)) for sz, os_, ts_ in zip(img.GetSize(), img.GetSpacing(), target_spacing)]
    resampler = sitk.ResampleImageFilter()
    resampler.SetOutputSpacing(target_spacing)
    resampler.SetSize(new_size)
    resampler.SetInterpolator(sitk.sitkNearestNeighbor if is_mask else sitk.sitkLinear)
    resampler.SetDefaultPixelValue(0 if is_mask else -1024)
    out = sitk.GetArrayFromImage(resampler.Execute(img))
    return out.astype(np.uint8) if is_mask else out.astype(np.float32)

def window_historical(hu):          # Week 10g: W150/C50 -> -25..125 HU -> 0..255
    lo, hi = 50 - 150 / 2, 50 + 150 / 2
    return ((np.clip(hu, lo, hi) - lo) / (hi - lo) * 255).astype(np.float32)

def window_corrected(hu):           # PLC-CECT release convention: -200..200 HU -> 0..255
    lo, hi = -200.0, 200.0
    return ((np.clip(hu, lo, hi) - lo) / (hi - lo) * 255).astype(np.float32)

WINDOWS = {'historical_W150C50': window_historical, 'corrected_-200_200': window_corrected}

def crop_96_around_tumor(vol, tumor_union, size=CROP_SIZE):   # Week 10g
    if tumor_union.sum() == 0:
        cz, cy, cx = vol.shape[0] // 2, vol.shape[1] // 2, vol.shape[2] // 2
    else:
        cz, cy, cx = np.argwhere(tumor_union > 0).mean(axis=0).astype(int)
    dz, dy, dx = [s // 2 for s in size]
    out = np.zeros(size, dtype=vol.dtype)
    z0, z1 = max(0, cz - dz), min(vol.shape[0], cz + dz)
    y0, y1 = max(0, cy - dy), min(vol.shape[1], cy + dy)
    x0, x1 = max(0, cx - dx), min(vol.shape[2], cx + dx)
    oz0 = dz - (cz - z0); oy0 = dy - (cy - y0); ox0 = dx - (cx - x0)
    out[oz0:oz0 + (z1 - z0), oy0:oy0 + (y1 - y0), ox0:ox0 + (x1 - x0)] = vol[z0:z1, y0:y1, x0:x1]
    return out, (cz, cy, cx)

def zscore_nonzero(x):                                          # Week 10g / Week 4
    mask = x > 0
    if mask.sum() < 100:
        return x.astype(np.float32)
    vals = x[mask]
    out = np.zeros_like(x, dtype=np.float32)
    out[mask] = (vals - vals.mean()) / (vals.std() + 1e-6)
    return out

def read_nifti(path):
    if not isinstance(path, str) or not path.strip() or path.strip().lower() in {'nan', 'none', 'null'}:
        return None, None
    img = sitk.ReadImage(path)
    return sitk.GetArrayFromImage(img).astype(np.float32), img.GetSpacing()

def preprocess_patient(row):
    """Week 10g preprocess_patient, split so the (slow) resampling is done once and both windows reuse it."""
    ct_r, tm_r, lm_r, spacing = {}, {}, {}, None
    raw = {}
    for phase in PHASE_ORDER:
        ct, sp = read_nifti(row.get(f'ct_{phase}'))
        if ct is not None and spacing is None:
            spacing = sp
        raw[phase] = (ct, read_nifti(row.get(f'mask_{phase}'))[0], read_nifti(row.get(f'liver_mask_{phase}'))[0])
    spacing = spacing or (1.0, 1.0, 1.0)
    for phase, (ct, tm, lm) in raw.items():
        ct_r[phase] = None if ct is None else resample_to_isotropic(ct, spacing)
        tm_r[phase] = None if (ct is None or tm is None) else resample_to_isotropic((tm > 0.5).astype(np.uint8), spacing, is_mask=True)
        lm_r[phase] = None if (ct is None or lm is None) else resample_to_isotropic((lm > 0.5).astype(np.uint8), spacing, is_mask=True)
    masks_present = [m for m in tm_r.values() if m is not None]
    if not masks_present:
        return None
    ref_shape = masks_present[0].shape
    tumor_union = np.zeros(ref_shape, dtype=np.uint8)
    for m in masks_present:
        if m.shape == ref_shape:
            tumor_union = np.maximum(tumor_union, m)
    liver = None
    for phase in ['C2', 'C1', 'P', 'C3']:
        lm = lm_r.get(phase)
        if lm is not None and lm.shape == ref_shape and lm.sum() > 0:
            liver = lm
            break
    if liver is None:
        liver = binary_dilation(tumor_union > 0, iterations=TUMOR_DILATION_MM).astype(np.uint8)
    out = {'tumor_voxels': int(tumor_union.sum()), 'available': [], 'tumor': np.zeros((4, *CROP_SIZE), np.uint8)}
    for window_name, window_fn in WINDOWS.items():
        cnn = np.zeros((4, *CROP_SIZE), dtype=np.float32)
        rad = np.zeros((4, *CROP_SIZE), dtype=np.float32)
        available = []
        for i, phase in enumerate(PHASE_ORDER):
            if ct_r[phase] is None or ct_r[phase].shape != ref_shape:
                continue
            crop, _ = crop_96_around_tumor(window_fn(ct_r[phase]) * (liver > 0), tumor_union)
            rad[i] = crop
            cnn[i] = zscore_nonzero(crop)
            available.append(phase)
            if tm_r[phase] is not None:
                out['tumor'][i] = crop_96_around_tumor(tm_r[phase], tumor_union)[0]
        out[window_name] = {'cnn': cnn, 'rad': rad}
        out['available'] = available
    return out

PRE = {}
t0 = time.time()
for n, (_, row) in enumerate(manifest.iterrows(), start=1):
    result = preprocess_patient(row)
    if result is None:
        raise RuntimeError(f'STOP — {row["patient_id"]} could not be preprocessed.')
    PRE[str(row['patient_id'])] = result
    if n % 10 == 0:
        print(f'  preprocessed {n}/103 ({(time.time() - t0) / 60:.1f} min)')
print(f'Preprocessing done: {len(PRE)} patients in {(time.time() - t0) / 60:.1f} min')

In [ ]:
# Cell 5 — CNN inference (single-phase, original four-phase, phase-dropout) and the reproduction gate
def make_cnn(in_channels):
    return resnet18(spatial_dims=3, n_input_channels=in_channels, num_classes=3, feed_forward=True)

def load_models(keys, in_channels):
    models = []
    for key in keys:
        checkpoint = torch.load(MODEL_FILES[key], map_location=DEVICE, weights_only=False)
        model = make_cnn(in_channels).to(DEVICE)
        model.load_state_dict(checkpoint['state_dict'])
        model.eval()
        models.append(model)
    return models

CNNS = {'single_phase_CNN': (load_models([f'w3_{f}' for f in range(5)], 1), [2]),
        'original_CNN': (load_models([f'w4_{f}' for f in range(5)], 4), [0, 1, 2, 3]),
        'phase_dropout_CNN': (load_models([f'd_{f}' for f in range(5)], 4), [0, 1, 2, 3])}

@torch.no_grad()
def cnn_probs(models, channels, window_name):
    rows = {}
    for pid in ALL_IDS:
        x = torch.from_numpy(PRE[pid][window_name]['cnn'][channels]).unsqueeze(0).to(DEVICE)
        rows[pid] = np.mean([torch.softmax(m(x), dim=1).float().cpu().numpy()[0] for m in models], axis=0)
    return pd.DataFrame.from_dict(rows, orient='index', columns=['p_HCC', 'p_ICC', 'p_CHCC'])

PROBS = {}   # (model, window) -> DataFrame indexed by patient
for name, (models, channels) in CNNS.items():
    for window_name in WINDOWS:
        PROBS[(name, window_name)] = cnn_probs(models, channels, window_name)

# Reproduction gate: original settings must reproduce the saved external CNN predictions
hist = pd.read_csv(INPUTS['historical_external'][0]).set_index('patient_id')
gate = {}
for name, prefix in [('original_CNN', 'w4'), ('single_phase_CNN', 'w3')]:
    ours = PROBS[(name, 'historical_W150C50')].loc[hist.index].to_numpy()
    saved = hist[[f'{prefix}_p_HCC', f'{prefix}_p_ICC', f'{prefix}_p_CHCC']].to_numpy()
    gate[name] = {'max_prob_diff': float(np.abs(ours - saved).max()),
                  'hcc_count_ours': int((ours.argmax(1) == 0).sum()), 'hcc_count_saved': int((saved.argmax(1) == 0).sum())}
REPRODUCED = all(g['max_prob_diff'] < 0.02 and g['hcc_count_ours'] == g['hcc_count_saved'] for g in gate.values())
print('Reproduction gate:', json.dumps(gate, indent=1))
if not REPRODUCED:
    if IN_COLAB:
        raise RuntimeError('STOP — the original external pipeline was not reproduced. Send this output for review.')
    print('LOCAL TEST: reproduction not expected with synthetic data.')
else:
    print('Reproduction PASS: the submitted external CNN results are reproduced (four-phase 0/103, single-phase 5/103).')

In [ ]:
# Cell 6 — Radiomics without shape features (E2 models), corrected window, Week-10g extraction settings
extractor = featureextractor.RadiomicsFeatureExtractor(binWidth=5)
extractor.disableAllImageTypes()
extractor.enableImageTypeByName('Original')
extractor.disableAllFeatures()
for feature_class in ('shape', 'firstorder', 'glcm', 'glrlm', 'glszm', 'gldm', 'ngtdm'):
    extractor.enableFeatureClassByName(feature_class)

def extract(pid, window_name):
    features = {}
    for i, phase in enumerate(PHASE_ORDER):
        if phase not in PRE[pid]['available'] or PRE[pid]['tumor'][i].sum() == 0:
            continue
        ct_img = sitk.GetImageFromArray(PRE[pid][window_name]['rad'][i].astype(np.float32)); ct_img.SetSpacing((1.0, 1.0, 1.0))
        mask_img = sitk.GetImageFromArray(PRE[pid]['tumor'][i].astype(np.uint8)); mask_img.SetSpacing((1.0, 1.0, 1.0))
        try:
            for key, value in extractor.execute(ct_img, mask_img).items():
                if not key.startswith('diagnostics'):
                    try:
                        features[f'{phase}_{key}'] = float(value)
                    except (TypeError, ValueError):
                        pass
        except Exception as error:
            print(f'  radiomics failed for {pid}/{phase}: {error}')
    return features

t0 = time.time()
ext_rad = pd.DataFrame.from_dict({pid: extract(pid, 'corrected_-200_200') for pid in ALL_IDS}, orient='index').reindex(ALL_IDS)
ext_rad.to_csv(OUT_DIR / 'E5_external_radiomics_features_corrected_window.csv')
print(f'Radiomics extracted: {ext_rad.shape} in {(time.time() - t0) / 60:.1f} min')

internal = pd.read_parquet(INPUTS['radiomics_features'][0])
internal['patient_id'] = internal['patient_id'].astype(str)
split = json.loads(INPUTS['split'][0].read_text())
dev_ids, test_ids = [str(x) for x in split['dev_ids']], [str(x) for x in split['test_ids']]
fold_splits = [json.loads(INPUTS[f'fold_{f}'][0].read_text()) for f in range(5)]
rad_models = [joblib.load(MODEL_FILES[f'rad_{f}']) for f in range(5)]

# Integrity: the saved E2 models must reproduce their saved internal test predictions
pool = rad_models[0]['feature_pool']
X_int_test = internal.set_index('patient_id').loc[test_ids, pool].to_numpy(dtype=np.float32)
p_int = np.mean([m['clf'].predict_proba(m['scaler'].transform(m['imputer'].transform(X_int_test[:, m['sel_idx']])))
                 for m in rad_models], axis=0)
saved_int = pd.read_csv(INPUTS['e2_noshape_test'][0]).set_index('patient_id').loc[test_ids, ['p_HCC', 'p_ICC', 'p_CHCC']].to_numpy()
rad_integrity = float(np.abs(p_int - saved_int).max())
print(f'E2 radiomics model integrity: max difference vs saved internal predictions {rad_integrity:.1e}')
if IN_COLAB and rad_integrity > 1e-6:
    raise RuntimeError('STOP — E2 radiomics models do not reproduce their saved predictions.')

def radiomics_probs(features, mode='real_plus_median'):
    per_fold, imputed, total = [], 0, 0
    for f, m in enumerate(rad_models):
        names = m['sel_names']
        train_ids = [p for p in dev_ids if p not in set(fold_splits[f]['val_ids'])]
        medians = internal.set_index('patient_id').loc[train_ids, names].median(axis=0, skipna=True)
        X = np.zeros((len(ALL_IDS), len(names)), dtype=np.float32)
        for r, pid in enumerate(ALL_IDS):
            for c, name in enumerate(names):
                value = features.loc[pid, name] if (mode == 'real_plus_median' and name in features.columns) else np.nan
                if np.isfinite(value):
                    X[r, c] = value
                else:
                    X[r, c] = medians[name]
                    imputed += 1
                total += 1
        per_fold.append(m['clf'].predict_proba(m['scaler'].transform(m['imputer'].transform(X))))
    return pd.DataFrame(np.mean(per_fold, axis=0), index=ALL_IDS, columns=['p_HCC', 'p_ICC', 'p_CHCC']), imputed / total

PROBS[('radiomics_noshape', 'corrected_-200_200')], IMPUTED_FRACTION = radiomics_probs(ext_rad)
PROBS[('radiomics_noshape_MEDIANS_ONLY_control', 'corrected_-200_200')], _ = radiomics_probs(ext_rad, mode='medians_only')
print(f'Radiomics: {IMPUTED_FRACTION:.1%} of selected-feature cells imputed with internal training medians')

In [ ]:
# Cell 7 — Fusion refitted on internal out-of-fold predictions, applied with observed and scenario clinical data
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
W4 = ['w4_p_HCC', 'w4_p_ICC', 'w4_p_CHCC']
W5 = ['w5_p_HCC', 'w5_p_ICC', 'w5_p_CHCC']
FUSIONS = {'cnn_plus_radiomics': W4 + W5, 'full_fusion': W4 + W5 + ['age', 'gender']}

oof_original = pd.read_csv(INPUTS['e2_oof_meta'][0], dtype={'patient_id': str})
d_pred = pd.read_csv(INPUTS['e3_d_predictions'][0], dtype={'patient_id': str})
d_oof = d_pred.query("pattern == 'complete' and split == 'oof'").set_index('patient_id')[['p_HCC', 'p_ICC', 'p_CHCC']]
oof_dropout = oof_original.copy()
oof_dropout[W4] = d_oof.loc[oof_dropout['patient_id']].to_numpy()
OOF_BY_CNN = {'original_CNN': oof_original, 'phase_dropout_CNN': oof_dropout}

def prepare(X):
    X = X.astype(np.float32)
    X[X <= -1] = np.nan
    return X

def fit_meta_folds(columns, oof):
    fitted = []
    for f in range(5):
        train = oof['fold'].astype(int).ne(f)
        X = prepare(oof.loc[train, columns].to_numpy())
        imputer = SimpleImputer(strategy='median').fit(X)
        scaler = StandardScaler().fit(imputer.transform(X))
        model = LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced', random_state=SEED)
        fitted.append((imputer, scaler, model.fit(scaler.transform(imputer.transform(X)), oof.loc[train, 'label'].astype(int))))
    return fitted

CLINICAL_SCENARIOS = {
    'observed_TCIA': (clinical_ext.set_index('patient_id').loc[ALL_IDS, 'age'].to_numpy(),
                      clinical_ext.set_index('patient_id').loc[ALL_IDS, 'sex'].to_numpy()),
    'both_missing_fold_median': (np.full(len(ALL_IDS), np.nan), np.full(len(ALL_IDS), np.nan)),
    'age_missing_sex_0': (np.full(len(ALL_IDS), np.nan), np.zeros(len(ALL_IDS))),
}
window = 'corrected_-200_200'
for cnn_name, oof in OOF_BY_CNN.items():
    for fusion_name, columns in FUSIONS.items():
        fitted = fit_meta_folds(columns, oof)
        base = pd.DataFrame(index=ALL_IDS)
        base[W4] = PROBS[(cnn_name, window)].loc[ALL_IDS].to_numpy()
        base[W5] = PROBS[('radiomics_noshape', window)].loc[ALL_IDS].to_numpy()
        scenarios = CLINICAL_SCENARIOS if fusion_name == 'full_fusion' else {'none': None}
        for scenario, values in scenarios.items():
            frame = base.copy()
            if values is not None:
                frame['age'], frame['gender'] = values
            X = prepare(frame[columns].to_numpy())
            probs = np.mean([model.predict_proba(scaler.transform(imputer.transform(X))) for imputer, scaler, model in fitted], axis=0)
            label = f'{fusion_name} [{cnn_name}]' + ('' if scenario == 'none' else f' {{{scenario}}}')
            PROBS[(label, window)] = pd.DataFrame(probs, index=ALL_IDS, columns=['p_HCC', 'p_ICC', 'p_CHCC'])
print('Fusion models applied:', sum(1 for k in PROBS if k[0].startswith(('full_fusion', 'cnn_plus'))))

In [ ]:
# Cell 8 — Results (exact intervals), saved outputs and the block to send back
def exact_ci(k, n):
    low = 0.0 if k == 0 else float(beta.ppf(0.025, k, n - k + 1))
    high = 1.0 if k == n else float(beta.ppf(0.975, k + 1, n - k))
    return low, high

SUBSETS = {'clean_89': CLEAN_IDS, 'all_103': ALL_IDS}
rows = []
for (model, window_name), probs in PROBS.items():
    for subset, ids in SUBSETS.items():
        p = probs.loc[ids].to_numpy()
        pred = p.argmax(axis=1)
        k, n = int((pred == 0).sum()), len(ids)
        low, high = exact_ci(k, n)
        rows.append({'model': model, 'window': window_name, 'subset': subset, 'n': n, 'hcc_correct': k,
                     'hcc_sensitivity': k / n, 'ci_low': low, 'ci_high': high,
                     'pred_HCC': k, 'pred_ICC': int((pred == 1).sum()), 'pred_CHCC': int((pred == 2).sum()),
                     'mean_p_HCC': float(p[:, 0].mean())})
results = pd.DataFrame(rows)
results.to_csv(OUT_DIR / 'E5_results.csv', index=False)

long = []
for (model, window_name), probs in PROBS.items():
    t = probs.copy()
    t.insert(0, 'patient_id', t.index)
    t.insert(0, 'window', window_name)
    t.insert(0, 'model', model)
    t['mixed_scan_patient'] = t['patient_id'].isin(MIXED_IDS)
    long.append(t)
pd.concat(long, ignore_index=True).to_csv(OUT_DIR / 'E5_predictions.csv', index=False)

summary = {'run_id': RUN_ID, 'notebook': 'E5_external_HCC_TACE_Seg_repaired_v2.ipynb', 'versions': VERSIONS,
           'input_sha256': input_hashes, 'reproduction_gate': gate, 'reproduced': REPRODUCED,
           'radiomics_model_integrity_max_diff': rad_integrity, 'imputed_fraction_corrected': IMPUTED_FRACTION,
           'mixed_scan': {'inputs': len(mixed), 'patients': sorted(MIXED_IDS)}, 'clean_n': len(CLEAN_IDS),
           'clinical': {'sheet': sheet_name, 'id_column': str(id_col), 'age_column': str(age_cols[0]),
                        'sex_column': str(sex_cols[0]), 'sex_coding': SEX_CODING, 'age_observed': int(clinical_ext['age'].notna().sum()),
                        'sex_observed': int(clinical_ext['sex'].notna().sum()), 'source': CLINICAL_URL},
           'protocol': {'windows': list(WINDOWS), 'primary': 'corrected window, clean_89, phase_dropout_CNN and full fusion with observed clinical data',
                        'decision_rule': 'argmax', 'no_external_tuning': True}}
(OUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2, default=str))
manifest_out = [{'file': p.relative_to(OUT_DIR).as_posix(), 'bytes': p.stat().st_size, 'sha256': sha256_of(p)}
                for p in sorted(OUT_DIR.rglob('*')) if p.is_file()]
pd.DataFrame(manifest_out).to_csv(OUT_DIR / 'output_manifest.csv', index=False)

show = results[['model', 'window', 'subset', 'hcc_correct', 'n', 'hcc_sensitivity', 'ci_low', 'ci_high',
                'pred_HCC', 'pred_ICC', 'pred_CHCC', 'mean_p_HCC']].copy()
show['window'] = show['window'].map({'historical_W150C50': 'historical', 'corrected_-200_200': 'corrected'})
print('\n================ COPY FROM HERE AND SEND BACK ================')
print(f'Run E5_{RUN_ID} | reproduced: {REPRODUCED} | radiomics integrity diff: {rad_integrity:.1e} | '
      f'imputed cells: {IMPUTED_FRACTION:.1%} | mixed-scan patients: {len(MIXED_IDS)} | clean n: {len(CLEAN_IDS)}')
print(f'Clinical: age observed {summary["clinical"]["age_observed"]}/103, sex observed {summary["clinical"]["sex_observed"]}/103 '
      f'(columns: {summary["clinical"]["age_column"]!r}, {summary["clinical"]["sex_column"]!r})')
print('Sex coding:', SEX_CODING)
print('Reproduction gate:', gate)
for subset in ['clean_89', 'all_103']:
    print(f'\n--- {subset} ---')
    print(show[show['subset'] == subset].drop(columns='subset').round(3).to_string(index=False))
print('\nVersions:', VERSIONS)
print('==============================================================')